In [2]:
%pip install datasets transformers

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
%pip install --upgrade datasets huggingface_hub fsspec

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [4]:
import sys
import os

sys.path.append("/home/jupyter/project/TverskyAttention/")

In [5]:
import math

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset

import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import GPT2TokenizerFast

from tqdm.auto import tqdm

In [86]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_head: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(d_model, 3 * d_model, bias=True)
        # output projection
        self.c_proj = nn.Linear(d_model, d_model, bias=True)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.n_head = n_head
        self.n_embd = d_model
        self.dropout = dropout
        # flash attention make GPU go brrrrr but support is only in PyTorch >= 2.0
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            print("WARNING: using slow attention. Flash Attention requires PyTorch >= 2.0")
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                        .view(1, 1, config.block_size, config.block_size))

    def forward(self, x: torch.Tensor):
        B, T, C = x.size() # batch size, sequence length, d_model

        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        if self.flash:
            y = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        y = self.resid_dropout(self.c_proj(y))
        return y

class BaselineBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        self.ln_1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model=d_model, n_head=num_heads)
        self.ln_2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(0.1)
        )

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class BaselineGPT(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, num_heads: int, num_layers: int, max_seq_len: int):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        
        self.blocks = nn.ModuleList([
            BaselineBlock(d_model, num_heads) for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Weight tying (связывание весов эмбеддингов и финального слоя)
        self.lm_head.weight = self.token_emb.weight

    def forward(self, idx: torch.Tensor, targets: torch.Tensor = None):
        B, T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=idx.device).to(torch.bool)
        
        x = self.token_emb(idx) + self.pos_emb(pos)
        
        for block in self.blocks:
            x = block(x, mask=mask)
            
        x = self.ln_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            # Считаем CrossEntropy (внутри применяется softmax)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            
        return logits, loss

In [87]:
class TextChunkDataset(IterableDataset):
    def __init__(self, tokenized_text, seq_len: int):
        self.tokens = tokenized_text
        self.seq_len = seq_len
        
    def __len__(self) -> int:
        return (len(self.tokens) - self.seq_len) // self.seq_len

    def __iter__(self):
        for i in range(0, len(self.tokens) - self.seq_len, self.seq_len):
            chunk = self.tokens[i:i + self.seq_len + 1]
            x = torch.tensor(chunk[:-1], dtype=torch.long)
            y = torch.tensor(chunk[1:], dtype=torch.long)
            yield x, y

def get_dataloaders(seq_len: int, batch_size: int) -> tuple[DataLoader, DataLoader, int]:
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
    tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
    
    def encode_split(split_name: str):
        text = "\n".join(dataset[split_name]['text'])
        return tokenizer.encode(text)
    
    train_tokens = encode_split("train")
    val_tokens = encode_split("validation")
    
    train_ds = TextChunkDataset(train_tokens, seq_len)
    val_ds = TextChunkDataset(val_tokens, seq_len)
    
    train_dl = DataLoader(train_ds, batch_size=batch_size)
    val_dl = DataLoader(val_ds, batch_size=batch_size)
    
    return train_dl, val_dl, tokenizer.vocab_size

In [88]:
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    steps = 0
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            _, loss = model(x, targets=y)
            total_loss += loss.item()
            steps += 1
            
    avg_loss = total_loss / steps
    perplexity = math.exp(avg_loss) # Перевод Loss в Перплексию
    model.train()
    return avg_loss, perplexity

In [89]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [90]:
seq_len = 128
batch_size = 32
d_model = 256
num_heads = 8
num_layers = 4
epochs = 10
lr = 5e-4

In [52]:
train_dl, val_dl, vocab_size = get_dataloaders(seq_len, batch_size)

Token indices sequence length is longer than the specified maximum sequence length for this model (2403644 > 1024). Running this sequence through the model will result in indexing errors


In [91]:
model = BaselineGPT(
    vocab_size=vocab_size, 
    d_model=d_model,
    num_heads=num_heads,
    num_layers=num_layers, 
    max_seq_len=seq_len
).to(device)

In [92]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Размер словаря: {vocab_size}")
print(f"Всего параметров в Baseline: {total_params:,}")

Размер словаря: 50257
Всего параметров в Baseline: 16,058,112


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
loss_history: list[float] = []
for epoch in range(epochs):
    total_train_loss = 0
    progress = tqdm(train_dl, desc=f"Эпоха {epoch+1}/{epochs}")

    for x, y in progress:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        _, loss = model(x, targets=y)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_train_loss += loss.item()
        progress.set_postfix({'loss': f"{loss.item():.4f}"})

    # Оценка на валидации
    val_loss, val_ppl = evaluate(model, val_dl, device)
    loss_history.append(val_loss)
    print(f"Эпоха {epoch+1} завершена | Train Loss: {total_train_loss/len(train_dl):.4f} | Val Loss: {val_loss:.4f} | Val Perplexity: {val_ppl:.2f}")

Эпоха 1/10: 100%|██████████| 587/587 [00:31<00:00, 18.59it/s, loss=7.9074] 


Эпоха 1 завершена | Train Loss: 15.5494 | Val Loss: 8.1337 | Val Perplexity: 3407.49


Эпоха 2/10: 100%|██████████| 587/587 [00:31<00:00, 18.62it/s, loss=7.0212]


Эпоха 2 завершена | Train Loss: 7.5657 | Val Loss: 7.2584 | Val Perplexity: 1420.00


Эпоха 3/10: 100%|██████████| 587/587 [00:31<00:00, 18.61it/s, loss=6.7028]


...

Эпоха 8/10: 100%|██████████| 587/587 [00:31<00:00, 18.61it/s, loss=6.1246]


Эпоха 8 завершена | Train Loss: 6.4151 | Val Loss: 6.5864 | Val Perplexity: 725.18


Эпоха 9/10: 100%|██████████| 587/587 [00:31<00:00, 18.72it/s, loss=6.0204]


Эпоха 9 завершена | Train Loss: 6.3226 | Val Loss: 6.5384 | Val Perplexity: 691.17


Эпоха 10/10:   0%|          | 2/587 [00:00<00:30, 18.93it/s, loss=6.4436]

In [ ]:
plt.plot(loss_history)
plt.title("Baseline validation loss")
plt.savefig("../thesis/resources/baseline_loss.png")

In [ ]:
torch.save(model.state_dict(), '../models/baseline.pth')